# Classifieur de genre

Les genres du catalogue sont desormais ceux de Deezer (genre de l'album), pour
les morceaux de 2021 comme pour les morceaux ajoutes en 2026, plus Country que
Deezer ne renseigne pas. Ce notebook choisit ces genres, les ecrit dans
`data/df_recommendation.csv`, les place sur un cercle (deux genres voisins
sont proches, Pop au centre), puis compare des modeles qui predisent le genre
a partir des descripteurs audio.

1. Chargement
2. Descriptif
3. Genres Deezer
4. Ecriture dans le CSV
5. Le cercle des genres
6. Modeles supervises : kNN, SVC, regression logistique, foret aleatoire, gradient boosting
7. Prediction des morceaux sans genre

Le code se lit de haut en bas, sans fonctions : modifier une cellule, puis
relancer les suivantes. Seule la partie 4 ecrit dans le CSV.

In [ ]:
import glob
import os
import sys

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

## 1. Chargement

In [ ]:
df = pd.read_csv("../data/df_recommendation.csv")

# Genres de 2021 (77 « genre seeds » Spotify et leurs 15 familles), gardes a part
# depuis que le CSV porte les genres Deezer. Vides pour les morceaux ajoutes.
genres_2021 = pd.read_csv("../data/genres_2021.csv", index_col="row")
df["genre_2021"] = genres_2021["Main_Genre"]
df["famille_2021"] = genres_2021["family_genre"]

# Playlist d'origine et date de sortie des morceaux ajoutes, un fichier par collecte :
# new_tracks.csv pour la derniere, new_tracks_<date>.csv pour les precedentes.
# Ces infos sont dans les sorties du pipeline (hors depot), pas dans le CSV.
fichiers = glob.glob(os.environ["LOCALAPPDATA"] + r"\museme\out\new_tracks*.csv")
pistes = pd.concat([pd.read_csv(f) for f in fichiers]).drop_duplicates("Preview")
df = df.join(pistes.set_index("Preview")[["playlist", "release_date"]], on="Preview")

# Les 3 803 premieres lignes viennent du catalogue de 2021, les suivantes du pipeline de 2026
df["origine"] = "2021"
df.loc[3803:, "origine"] = "2026"
anciens = df.iloc[:3803].copy()
nouveaux = df.iloc[3803:].copy()

# Les 63 descripteurs audio, de chroma_stft_mean a onset_var
descripteurs = df.loc[:, "chroma_stft_mean":"onset_var"].columns

print(len(anciens), "morceaux de 2021 ;", len(nouveaux), "morceaux ajoutes ;", len(descripteurs), "descripteurs")

In [ ]:
nouveaux[["Artist", "Song", "playlist", "release_date"]].head()

## 2. Descriptif

### 2.1 Les genres de 2021

In [ ]:
genres = anciens["genre_2021"].value_counts()
print(len(genres), "genres")
genres

In [ ]:
# Chaque genre appartient a une seule famille
familles = anciens.groupby("famille_2021")["genre_2021"].agg(morceaux="size", genres="nunique", liste="unique")
familles.sort_values("morceaux", ascending=False)

In [ ]:
# Quelques morceaux d'un genre, pour juger les etiquettes : changer GENRE et relancer
GENRE = "pop"
anciens.loc[anciens["genre_2021"] == GENRE, ["Artist", "Song", "Album"]].head(20)

### 2.2 Doublons : un meme morceau sous plusieurs genres

In [ ]:
morceau = anciens["Artist"] + " - " + anciens["Song"]
doublons = anciens[morceau.duplicated(keep=False)].sort_values(["Artist", "Song"])
print(morceau.duplicated().sum(), "lignes en double ;", morceau[morceau.duplicated()].nunique(), "morceaux concernes")
doublons[["Artist", "Song", "genre_2021", "famille_2021"]]

### 2.3 Artistes : un meme artiste sous plusieurs genres

In [ ]:
artistes = anciens.groupby("Artist")["genre_2021"].agg(morceaux="size", nb_genres="nunique", genres="unique")
print(len(artistes), "artistes ;", (artistes["nb_genres"] > 1).sum(), "sous plusieurs genres")
artistes.sort_values("morceaux", ascending=False).head(30)

### 2.4 Les morceaux ajoutes

In [ ]:
nouveaux["playlist"].value_counts()

In [ ]:
# Annee de sortie
nouveaux["release_date"].str[:4].value_counts().sort_index()

In [ ]:
# Morceaux ajoutes dont l'artiste est deja au catalogue de 2021 : son genre de 2021 est un indice
connus = nouveaux[nouveaux["Artist"].isin(anciens["Artist"])]
print(len(connus), "morceaux ajoutes sur", len(nouveaux), "ont un artiste deja present en 2021")
connus[["Artist", "Song", "playlist"]].merge(artistes["genres"], left_on="Artist", right_index=True).head(30)

### 2.5 Descripteurs : 2021 contre morceaux ajoutes

Un rapport loin de 1 signale un descripteur qui ne se comporte pas pareil sur
les deux groupes (volume, mixage...).

In [ ]:
comparaison = pd.DataFrame({
    "mediane_2021": anciens[descripteurs].median(),
    "mediane_ajoutes": nouveaux[descripteurs].median(),
})
comparaison["rapport"] = comparaison["mediane_ajoutes"] / comparaison["mediane_2021"]
comparaison.round(3)

## 3. Genres Deezer

`python -m pipeline deezer` cherche chaque morceau du CSV sur Deezer (API
publique, sans compte) et note les genres de son album. Le fichier est dans
les sorties du pipeline, hors depot, comme `new_tracks.csv`.

Un album peut avoir plusieurs genres : le premier est le genre principal, et
c'est lui qu'on garde. Deezer se trompe parfois et ne trouve pas tout. La
colonne `genre` part donc du genre Deezer, puis on la corrige et on la
complete. La colonne `source` note d'ou vient chaque genre.

### 3.1 Le genre principal

In [ ]:
deezer = pd.read_csv(os.environ["LOCALAPPDATA"] + r"\museme\out\deezer_genres.csv", index_col="row")

# L'index de deezer est le numero de ligne du CSV, comme celui de df
df["genre_deezer"] = deezer["deezer_genres"].str.split("|").str[0]

print(deezer["deezer_artist"].notna().sum(), "morceaux trouves sur Deezer sur", len(deezer), ";",
      df["genre_deezer"].notna().sum(), "avec un genre")
df["genre_deezer"].value_counts()

In [ ]:
# Deuxieme avis sur les familles de 2021 : leur genre Deezer
pd.crosstab(df["famille_2021"], df["genre_deezer"])

### 3.2 Nettoyage

In [ ]:
df["genre"] = df["genre_deezer"]
df["source"] = "deezer"

# Pas des genres musicaux : bandes originales (Lady Gaga, anime japonais) et albums pour
# enfants (Vivaldi, Justin Timberlake). Ces morceaux sont completes en 3.3, comme ceux que
# Deezer n'a pas trouves.
df.loc[df["genre"].isin(["Films/Games", "Kids"]), "genre"] = None

# Genres trop rares pour un modele (10 morceaux au plus), rattaches au genre le plus proche
REGROUPEMENT = {
    "Soul & Funk": "R&B",              # Sly & The Family Stone, Monophonics
    "Spirituality & Religion": "R&B",  # un morceau de gospel ; le gospel est en R&B sur Deezer
    "Singer & Songwriter": "Folk",     # Carole King, Creedence Clearwater Revival
}
df["genre"] = df["genre"].replace(REGROUPEMENT)

In [ ]:
# Erreurs de Deezer, d'apres ce que font ces artistes : tous leurs morceaux prennent ce genre.
# C'est l'ancienne liste FAUX_RAP (Deezer classe ces artistes en Rap/Hip Hop), plus des
# compositeurs que Deezer range hors du classique, et Sting.
CORRECTIONS = {
    "D-Block": "Dance",              # duo hardstyle, confondu avec un groupe de rap homonyme
    "Carl Thomas": "R&B",            # chanteur r&b
    "Little Beaver": "R&B",          # guitariste funk des annees 70
    "Two Fingers": "Electro",        # instrumentaux electro (Amon Tobin)
    "Blockhead": "Electro",          # instrumentaux hip-hop, sans rap
    "Matta": "Electro",              # dubstep
    "Gorillaz": "Alternative",       # Last Living Souls, Kids with Guns, Feel Good Inc.
    "Vybz Kartel": "Reggae",         # dancehall
    "Gente De Zona": "Latin Music",  # reggaeton cubain
    "Frédéric Chopin": "Classical",  # Nocturne op. 9 n° 2 en Electro (album « Chopin journal intime »)
    "John Cage": "Classical",        # 4'33" en Pop
    "George Gershwin": "Classical",  # Prelude No. 1 en Jazz
    "Sting": "Pop",                  # Englishman In New York trouve sur un album orchestral, en Classical
}
corrige = df["Artist"].isin(list(CORRECTIONS))
df.loc[corrige, "genre"] = df.loc[corrige, "Artist"].map(CORRECTIONS)
df.loc[corrige, "source"] = "correction"

# Erreurs de Deezer sur un seul morceau : les autres morceaux de l'artiste gardent leur genre
CORRECTIONS_MORCEAUX = {
    # album en Classical sur Deezer ; Pop, le genre le plus frequent de Pitbull
    ("Pitbull", "Greenlight (feat. Flo Rida & LunchMoney Lewis)"): "Pop",
    # Pop sur Deezer depuis que l'artiste est le compositeur (et non Zamfir) ; genre garde
    ("James Last", "The Lonely Shepherd"): "Classical",
}
for (artiste, titre), genre in CORRECTIONS_MORCEAUX.items():
    ligne = df["Artist"].eq(artiste) & df["Song"].eq(titre)
    assert ligne.sum() == 1, (artiste, titre)
    df.loc[ligne, "genre"] = genre
    df.loc[ligne, "source"] = "correction"
    corrige = corrige | ligne
print(corrige.sum(), "morceaux corriges")

In [ ]:
# Country : Deezer ne donne aucun genre (genre_id -1) a 97 des 149 morceaux country de 2021,
# et range la plupart des autres en Rock ou en Pop (Chris Knight, The Country Gentlemen, Dwight Yoakam).
# Les trois genres country de 2021 forment donc le genre Country, trouves sur Deezer ou non.
COUNTRY_2021 = ["country", "bluegrass", "honky-tonk"]
country = df["genre_2021"].isin(COUNTRY_2021)
df.loc[country, "genre"] = "Country"
df.loc[country, "source"] = "country_2021"
print(country.sum(), "morceaux en Country")

### 3.3 Morceaux sans genre

Deux recours, dans l'ordre : le genre de l'artiste, puis, pour les morceaux
de 2021, une correspondance entre genre de 2021 et genre Deezer.

In [ ]:
# 1) Un autre morceau du meme artiste a un genre : on prend le genre le plus frequent de l'artiste
genre_artiste = pd.crosstab(df["Artist"], df["genre"]).idxmax(axis=1)

sans_genre = df["genre"].isna()
df.loc[sans_genre, "genre"] = df.loc[sans_genre, "Artist"].map(genre_artiste)
df.loc[sans_genre & df["genre"].notna(), "source"] = "artiste"
print((df["source"] == "artiste").sum(), "morceaux completes par leur artiste ;",
      df["genre"].isna().sum(), "encore sans genre")

In [ ]:
# 2) Correspondance genre de 2021 -> genre Deezer : par defaut, le genre Deezer le plus
# frequent parmi les morceaux du meme genre de 2021 dont le genre vient de Deezer
base = df[df["source"].isin(["deezer", "correction"])]
comptes = pd.crosstab(base["genre_2021"], base["genre"])

correspondance = pd.DataFrame({
    "genre_deezer": comptes.idxmax(axis=1),
    "part": (comptes.max(axis=1) / comptes.sum(axis=1)).round(2),
    "morceaux_a_completer": df.loc[df["genre"].isna(), "genre_2021"].value_counts(),
})
correspondance["morceaux_a_completer"] = correspondance["morceaux_a_completer"].fillna(0).astype(int)

# Les genres de 2021 qui comptent : ceux qui ont des morceaux a completer
correspondance.sort_values("morceaux_a_completer", ascending=False).head(25)

In [ ]:
# Detail d'un genre de 2021 : changer GENRE_2021 et relancer
GENRE_2021 = "tango"
comptes.loc[GENRE_2021].sort_values(ascending=False).head(5)

In [ ]:
# Correspondances corrigees a la main, quand la majorite Deezer est faible ou contre-intuitive
CORRESPONDANCE_FORCEE = {
    "folk": "Folk",   # majorite Alternative (14 sur 40), mais Folk est un genre Deezer
    "soul": "R&B",    # Pop 20, R&B 18
    "ska": "Reggae",  # egalite Alternative 13, Reggae 13
}
for genre_2021, genre in CORRESPONDANCE_FORCEE.items():
    correspondance.loc[genre_2021, "genre_deezer"] = genre

a_completer = df["genre"].isna() & df["genre_2021"].notna()
df.loc[a_completer, "genre"] = df.loc[a_completer, "genre_2021"].map(correspondance["genre_deezer"])
df.loc[a_completer, "source"] = "genre_2021"
df.loc[df["genre"].isna(), "source"] = "aucune"

# D'ou vient le genre
pd.crosstab(df["source"], df["origine"])

### 3.4 Resultat

In [ ]:
# Les genres retenus
df["genre"].value_counts()

In [ ]:
# Morceaux ajoutes : genre par playlist d'origine
pd.crosstab(df["playlist"], df["genre"])

In [ ]:
# Morceaux restes sans genre : Deezer ne leur donne pas de genre et leur artiste n'en a pas.
# Beaucoup de country (Morgan Wallen, Zach Bryan, Luke Combs), genre que Deezer ne renseigne pas,
# et de musique mexicaine (Peso Pluma, Fuerza Regida). Le modele les predit en partie 7.
df.loc[df["genre"].isna(), ["Artist", "Song", "playlist"]]

In [ ]:
# Quelques morceaux, pour juger a l'oreille : changer PLAYLIST et relancer
PLAYLIST = "Rock Classics"
df.loc[df["playlist"] == PLAYLIST, ["Artist", "Song", "genre_deezer", "genre", "source"]].head(20)

## 4. Ecriture dans le CSV

`write_genres` (module `pipeline/genres.py`) ecrit `genre` dans les colonnes
`Main_Genre` et `family_genre` de `data/df_recommendation.csv`. Seuls ces deux
champs changent : les descripteurs restent identiques octet pour octet. Un
genre manquant laisse le champ vide. La cellule se relance sans risque apres
un changement plus haut.

L'app fait de `Main_Genre` des indicatrices dans son espace de similarite :
changer les genres change les recommandations de tout le catalogue.

In [ ]:
sys.path.append("..")
from pipeline.genres import write_genres

write_genres(df["genre"]);

## 5. Le cercle des genres

Les genres sont places sur un cercle, de sorte que deux voisins soient des
genres proches. Pop, proche de presque tout, est au centre : il est voisin de
tous les genres. Si le modele predit un voisin du vrai genre, l'erreur est
pardonnable.

L'ordre est un choix musical, verifie sur Deezer : quand un album porte
plusieurs genres (*Rock|Metal*), ces genres sont proches. Cette verification
ne doit rien aux descripteurs audio, donc rien au modele. Country n'a aucun
album sur Deezer dans nos donnees : sa place est un choix musical.

Attention : avec Pop au centre, repondre Pop n'est jamais une erreur
lointaine. Un modele qui repond Pop trop souvent gonfle son score « a un
voisin pres » : la partie 6 suit donc aussi la part de reponses Pop.

In [ ]:
# Au centre, voisin de tous les genres
CENTRE = "Pop"

# Sur le cercle, dans le sens des aiguilles d'une montre ; le dernier est voisin du premier.
# Le commentaire justifie le lien avec le genre de la ligne precedente.
CERCLE = [
    "Asian Music",      # (avec Dance, le dernier) j-dance, k-pop
    "R&B",              # ballades cantopop en R&B sur Deezer
    "African Music",    # afrobeat de 2021 : African Music, puis R&B sur Deezer
    "Reggae",           # dancehall et afrobeats se melangent
    "Rap/Hip Hop",      # albums Rap/Hip Hop + Reggae (dancehall)
    "Latin Music",      # reggaeton : albums Latin Music + Rap/Hip Hop
    "Brazilian Music",  # albums Latin Music + Brazilian Music
    "Jazz",             # bossa nova
    "Classical",        # instrumental, acoustique
    "Folk",             # acoustique
    "Country",          # folk americain, bluegrass
    "Alternative",      # alt-country ; Birdtalker (bluegrass) en Alternative sur Deezer
    "Rock",             # albums Rock + Alternative, le lien le plus frequent
    "Metal",            # albums Rock + Metal
    "Electro",          # industrial : famille metal en 2021, Electro sur Deezer
    "Dance",            # albums Electro + Dance
]
GENRES = [CENTRE] + CERCLE
print("genres hors cercle :", set(df["genre"].dropna()) - set(GENRES))

# Nombre de pas entre deux genres du cercle (de 0 a 8), par le plus court des deux sens
position = pd.Series(range(len(CERCLE)), index=CERCLE)
ecart = abs(position.to_numpy()[:, None] - position.to_numpy()[None, :])
distance = pd.DataFrame(np.minimum(ecart, len(CERCLE) - ecart), index=CERCLE, columns=CERCLE)

# Pop, au centre, est a un pas de chaque genre
distance = distance.reindex(index=GENRES, columns=GENRES, fill_value=1)
distance.loc[CENTRE, CENTRE] = 0
distance

In [ ]:
import plotly.graph_objects as go

# Coordonnees : les genres du cercle sur un cercle de rayon 1, Pop au centre
angle = 2 * np.pi * position / len(CERCLE)
x_genre = pd.concat([pd.Series({CENTRE: 0.0}), np.sin(angle)])
y_genre = pd.concat([pd.Series({CENTRE: 0.0}), np.cos(angle)])
morceaux = df["genre"].value_counts().reindex(GENRES)

# Le cercle : un trait relie deux voisins ; taille du point = nombre de morceaux
fig = go.Figure()
for a, b in zip(CERCLE, CERCLE[1:] + CERCLE[:1]):
    fig.add_trace(go.Scatter(x=[x_genre[a], x_genre[b]], y=[y_genre[a], y_genre[b]], mode="lines",
                             line=dict(width=4, color="#2a78d6"), hoverinfo="skip", showlegend=False))
for g in CERCLE:
    fig.add_trace(go.Scatter(x=[0, x_genre[g]], y=[0, y_genre[g]], mode="lines",
                             line=dict(width=1.5, color="#2a78d6", dash="dot"), hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(
    x=x_genre, y=y_genre, mode="markers",
    marker=dict(size=10 + np.sqrt(morceaux.to_numpy()), color="#0b0b0b", opacity=1, line=dict(width=3, color="#fcfcfb")),
    hoverinfo="text", hovertext=[f"{g} : {morceaux[g]} morceaux" for g in GENRES], showlegend=False,
))
# Noms : Pop dans son point, les autres a l'exterieur du cercle
fig.add_trace(go.Scatter(x=[0], y=[0], mode="text", text=[CENTRE], textfont=dict(color="#ffffff", size=14),
                         hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(
    x=1.15 * x_genre[CERCLE], y=1.15 * y_genre[CERCLE], mode="text", text=CERCLE, textfont=dict(size=14),
    textposition=["middle right" if u > 0.2 else "middle left" if u < -0.2 else "middle center" for u in x_genre[CERCLE]],
    hoverinfo="skip", showlegend=False,
))
fig.update_layout(
    title=dict(text="Le cercle des genres<br><sup>Deux genres relies sont voisins ; Pop, au centre, est voisin de tous. "
                    "Taille du point : nombre de morceaux.</sup>"),
    width=800, height=750, plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb", font=dict(color="#0b0b0b"),
    xaxis=dict(visible=False, range=[-1.8, 1.8]), yaxis=dict(visible=False, range=[-1.3, 1.3], scaleanchor="x"),
    margin=dict(l=20, r=20, t=90, b=20),
)
fig.show()

# Image du README (necessite le paquet kaleido et Chrome, absents des requirements)
# fig.write_image("../docs/cercle_genres.png", scale=2)

In [ ]:
# Verification sur Deezer : nombre d'albums qui portent a la fois les deux genres.
# Pop en premier (voisin de tous), puis l'ordre du cercle : les voisins sont juste a cote
# de la diagonale (et dans les coins du cercle).
albums = deezer.dropna(subset=["deezer_genres"]).drop_duplicates("deezer_album_id")
cooccurrences = pd.DataFrame(0, index=GENRES, columns=GENRES)
for genres_album in albums["deezer_genres"].str.split("|"):
    genres_album = [REGROUPEMENT.get(g, g) for g in genres_album]
    genres_album = [g for g in set(genres_album) if g in GENRES]
    for a in genres_album:
        for b in genres_album:
            if a != b:
                cooccurrences.loc[a, b] += 1
cooccurrences

In [ ]:
# Les 16 paires de voisins sur le cercle : celles sans album commun sont des choix musicaux
# (genres « du monde », acoustiques et Country, rares ou absents sur Deezer)
voisins = pd.DataFrame({"genre": CERCLE, "voisin_suivant": CERCLE[1:] + CERCLE[:1]})
voisins["albums_en_commun"] = [cooccurrences.loc[a, b] for a, b in zip(voisins["genre"], voisins["voisin_suivant"])]
voisins

In [ ]:
# Les albums Deezer sur le cercle : un trait par paire de genres portes par un meme album
# (3 albums au moins), en bleu entre voisins, en gris sinon
fig = go.Figure()
for a in GENRES:
    for b in GENRES:
        n = cooccurrences.loc[a, b]
        if GENRES.index(a) < GENRES.index(b) and n >= 3:
            voisin = distance.loc[a, b] == 1
            fig.add_trace(go.Scatter(
                x=[x_genre[a], x_genre[b]], y=[y_genre[a], y_genre[b]], mode="lines",
                line=dict(width=1 + n / 8, color="#2a78d6" if voisin else "#c9c8c2"),
                showlegend=False,
                hoverinfo="text", text=f"{a} + {b} : {n} albums",
            ))
fig.add_trace(go.Scatter(
    x=x_genre, y=y_genre, mode="markers",
    marker=dict(size=10 + np.sqrt(morceaux.to_numpy()), color="#0b0b0b", opacity=1, line=dict(width=2, color="#fcfcfb")),
    hoverinfo="text", hovertext=[f"{g} : {morceaux[g]} morceaux" for g in GENRES], showlegend=False,
))
fig.add_trace(go.Scatter(x=[0], y=[0], mode="text", text=[CENTRE], textfont=dict(color="#ffffff"),
                         hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(
    x=1.15 * x_genre[CERCLE], y=1.15 * y_genre[CERCLE], mode="text", text=CERCLE,
    textposition=["middle right" if u > 0.2 else "middle left" if u < -0.2 else "middle center" for u in x_genre[CERCLE]],
    hoverinfo="skip", showlegend=False,
))
# Legende : une entree par couleur de trait
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(width=3, color="#2a78d6"), name="voisins"))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(width=3, color="#c9c8c2"), name="non voisins"))
fig.update_layout(
    title="Albums Deezer qui portent deux genres",
    width=750, height=750, plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb", font=dict(color="#0b0b0b"),
    xaxis=dict(visible=False, range=[-1.7, 1.7]), yaxis=dict(visible=False, range=[-1.35, 1.35], scaleanchor="x"),
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.02),
)
fig.show()

## 6. Modeles supervises

Tous les morceaux qui ont un genre (2021 et ajoutes) sont coupes en deux avec
`train_test_split` : 80 % pour entrainer, 20 % pour tester sur des morceaux
jamais vus. Le decoupage est stratifie : chaque genre garde la meme
proportion dans les deux parts.

Quatre scores par modele, reunis dans le tableau `resultats` :
- `exact` : part de genres exacts ;
- `exact_equilibre` : la meme part, calculee genre par genre puis moyennee :
  chaque genre compte autant, quel que soit son nombre de morceaux ;
- `voisin` : part de genres exacts ou voisins sur le cercle ;
- `reponses_pop` : part des predictions qui sont Pop. Au-dela de la part de
  Pop dans le test, le modele se refugie dans Pop, ce qui gonfle `voisin`.

Le test compte 940 morceaux : un ecart de 1 ou 2 points entre deux modeles
n'est pas significatif.

In [ ]:
etiquetes = df[df["genre"].notna()]

# Un meme morceau present deux fois ne doit pas finir a la fois dans le train et dans le test
etiquetes = etiquetes.drop_duplicates(subset=["Artist", "Song"])

print(len(etiquetes), "morceaux etiquetes")
pd.crosstab(etiquetes["genre"], etiquetes["origine"]).reindex(GENRES)

In [ ]:
from sklearn.model_selection import train_test_split

X = etiquetes[descripteurs]
y = etiquetes["genre"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
print(len(X_train), "morceaux d'entrainement ;", len(X_test), "morceaux de test ;",
      round((y_test == CENTRE).mean(), 3), "de Pop dans le test")

In [ ]:
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import StandardScaler

# Centrage-reduction : moyennes et ecarts-types calcules sur le train seulement
scaler = StandardScaler()
scaler.fit(X_train)
X_train_reduit = scaler.transform(X_train)
X_test_reduit = scaler.transform(X_test)

# Scores de chaque modele, une ligne par modele (relancer une cellule remplace sa ligne)
resultats = {}

# Repere : toujours repondre Pop. Pop etant voisin de tous, il a 100 % « a un voisin pres »
pred_pop = np.full(len(y_test), CENTRE)
pas_pop = np.array([distance.loc[vrai, predit] for vrai, predit in zip(y_test, pred_pop)])
resultats["toujours Pop"] = {
    "exact": (pas_pop == 0).mean(),
    "exact_equilibre": balanced_accuracy_score(y_test, pred_pop),
    "voisin": (pas_pop <= 1).mean(),
    "reponses_pop": (pred_pop == CENTRE).mean(),
}
pd.DataFrame(resultats).T.round(3)

### 6.1 Plus proches voisins (kNN)

Le kNN donne a un morceau le genre le plus frequent parmi ses k morceaux les
plus proches. Avec la distance cosinus sur les descripteurs centres-reduits,
c'est la similarite de l'app, sans les indicatrices de genre : le genre
predit est celui des morceaux que l'app recommanderait.

On choisit k (et la distance) par validation croisee sur le train, en 5
parts, pour ne pas regler le modele sur le test.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.neighbors import KNeighborsClassifier

# Pour chaque distance et chaque k : genres predits par validation croisee sur le train, puis scores.
# weights="distance" : les voisins les plus proches comptent davantage dans le vote.
choix_k = []
for metrique in ["cosine", "euclidean"]:
    for k in [1, 5, 10, 15, 20, 30, 50, 80]:
        knn = KNeighborsClassifier(n_neighbors=k, metric=metrique, weights="distance")
        pred_cv = cross_val_predict(knn, X_train_reduit, y_train, cv=5)
        pas_cv = np.array([distance.loc[vrai, predit] for vrai, predit in zip(y_train, pred_cv)])
        choix_k.append({
            "distance": metrique,
            "k": k,
            "exact": (pas_cv == 0).mean(),
            "exact_equilibre": balanced_accuracy_score(y_train, pred_cv),
            "voisin": (pas_cv <= 1).mean(),
            "reponses_pop": (pred_cv == CENTRE).mean(),
        })
pd.DataFrame(choix_k).round(3)

In [ ]:
# kNN retenu d'apres la validation croisee : changer K et METRIQUE, puis relancer
K = 10
METRIQUE = "cosine"

knn = KNeighborsClassifier(n_neighbors=K, metric=METRIQUE, weights="distance")
knn.fit(X_train_reduit, y_train)
pred_knn = knn.predict(X_test_reduit)

pas_knn = np.array([distance.loc[vrai, predit] for vrai, predit in zip(y_test, pred_knn)])
resultats["kNN"] = {
    "exact": (pas_knn == 0).mean(),
    "exact_equilibre": balanced_accuracy_score(y_test, pred_knn),
    "voisin": (pas_knn <= 1).mean(),
    "reponses_pop": (pred_knn == CENTRE).mean(),
}
pd.DataFrame(resultats).T.round(3)

### 6.2 SVC

Machine a vecteurs de support a noyau gaussien : le modele des versions
precedentes du notebook.

In [ ]:
from sklearn.svm import SVC

# class_weight="balanced" : les petits genres (Folk, African Music : une quarantaine de morceaux)
# pesent autant que les grands (Pop, Rock : plus de 700)
svc = SVC(C=10, class_weight="balanced")
svc.fit(X_train_reduit, y_train)
pred_svc = svc.predict(X_test_reduit)

pas_svc = np.array([distance.loc[vrai, predit] for vrai, predit in zip(y_test, pred_svc)])
resultats["SVC"] = {
    "exact": (pas_svc == 0).mean(),
    "exact_equilibre": balanced_accuracy_score(y_test, pred_svc),
    "voisin": (pas_svc <= 1).mean(),
    "reponses_pop": (pred_svc == CENTRE).mean(),
}
pd.DataFrame(resultats).T.round(3)

### 6.3 Regression logistique

Modele lineaire : chaque genre a un score, somme ponderee des descripteurs.
Le plus simple des modeles, un repere pour les autres.

In [ ]:
from sklearn.linear_model import LogisticRegression

logistique = LogisticRegression(class_weight="balanced", max_iter=3000)
logistique.fit(X_train_reduit, y_train)
pred_logistique = logistique.predict(X_test_reduit)

pas_logistique = np.array([distance.loc[vrai, predit] for vrai, predit in zip(y_test, pred_logistique)])
resultats["regression logistique"] = {
    "exact": (pas_logistique == 0).mean(),
    "exact_equilibre": balanced_accuracy_score(y_test, pred_logistique),
    "voisin": (pas_logistique <= 1).mean(),
    "reponses_pop": (pred_logistique == CENTRE).mean(),
}
pd.DataFrame(resultats).T.round(3)

### 6.4 Foret aleatoire

500 arbres de decision, chacun entraine sur un tirage des morceaux et des
descripteurs ; le genre predit est celui du vote majoritaire.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

foret = RandomForestClassifier(n_estimators=500, class_weight="balanced", n_jobs=-1, random_state=0)
foret.fit(X_train_reduit, y_train)
pred_foret = foret.predict(X_test_reduit)

pas_foret = np.array([distance.loc[vrai, predit] for vrai, predit in zip(y_test, pred_foret)])
resultats["foret aleatoire"] = {
    "exact": (pas_foret == 0).mean(),
    "exact_equilibre": balanced_accuracy_score(y_test, pred_foret),
    "voisin": (pas_foret <= 1).mean(),
    "reponses_pop": (pred_foret == CENTRE).mean(),
}
pd.DataFrame(resultats).T.round(3)

### 6.5 Gradient boosting

Des arbres de decision construits l'un apres l'autre, chacun corrigeant les
erreurs des precedents. Le plus long a entrainer : environ une minute.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

boosting = HistGradientBoostingClassifier(class_weight="balanced", random_state=0)
boosting.fit(X_train_reduit, y_train)
pred_boosting = boosting.predict(X_test_reduit)

pas_boosting = np.array([distance.loc[vrai, predit] for vrai, predit in zip(y_test, pred_boosting)])
resultats["gradient boosting"] = {
    "exact": (pas_boosting == 0).mean(),
    "exact_equilibre": balanced_accuracy_score(y_test, pred_boosting),
    "voisin": (pas_boosting <= 1).mean(),
    "reponses_pop": (pred_boosting == CENTRE).mean(),
}
pd.DataFrame(resultats).T.round(3)

### 6.6 Comparaison

In [ ]:
pd.DataFrame(resultats).T.sort_values("exact_equilibre", ascending=False).round(3)

Sur ce decoupage (les chiffres bougent d'un point ou deux avec un autre `random_state`) :

- **Le SVC est le meilleur compromis**, et le modele retenu : presque le
  meilleur sur le genre exact (39 %), le meilleur en exact equilibre (43 %).
  Grace a `class_weight`, il reconnait aussi les petits genres, et il repond
  rarement Pop (8 %, pour 16 % de Pop dans le test).
- **Le gradient boosting** a le meilleur exact (40 %) et le meilleur « voisin »
  (80 %), mais il repond Pop une fois sur quatre : son score « voisin » vient
  en partie de la, et les petits genres en patissent (34 % en equilibre).
- **Le kNN** plafonne a 36 % d'exact et 35 % en equilibre. Sans poids par
  genre, le vote des voisins penche vers les grands genres (19 % de reponses
  Pop). Avec la distance euclidienne c'est pire : plus k grandit, plus il
  repond Pop, et plus « voisin » monte alors que l'exact baisse (partie 6.1).
- **La regression logistique**, modele lineaire, est la plus faible sur
  l'exact : les frontieres entre genres ne sont pas droites.

Le kNN cosinus dit aussi quelque chose de l'app : avec k = 1, il repond le
genre du morceau le plus proche, et tombe juste 31 % du temps (validation
croisee). Sans les indicatrices de genre, la premiere recommandation de l'app
ne serait donc du meme genre que le morceau qu'une fois sur trois environ.

### 6.7 Le modele retenu, en detail

In [ ]:
from sklearn.metrics import classification_report

# Modele examine dans la suite et en partie 7 : changer les deux lignes pour en examiner un autre
modele = svc
y_pred = pred_svc

pas = np.array([distance.loc[vrai, predit] for vrai, predit in zip(y_test, y_pred)])
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# A combien de pas du vrai genre tombent les predictions
pd.Series(pas).value_counts().sort_index()

In [ ]:
# Par genre : part de bonnes reponses, exactes et a un voisin pres
par_genre = pd.DataFrame({"genre": y_test.to_numpy(), "exact": pas == 0, "voisin": pas <= 1})
par_genre.groupby("genre")[["exact", "voisin"]].mean().reindex(GENRES).round(2)

In [ ]:
# Matrice de confusion : en ligne le vrai genre, en colonne le genre predit.
# Pop en premier, puis l'ordre du cercle : les erreurs pardonnables sont dans la ligne et la
# colonne Pop, et juste a cote de la diagonale.
pd.crosstab(y_test, y_pred, rownames=["vrai"], colnames=["predit"]).reindex(index=GENRES, columns=GENRES, fill_value=0)

In [ ]:
# Les erreurs sur le cercle : un trait par paire de genres souvent confondus (8 erreurs au moins,
# dans un sens ou dans l'autre), en bleu entre voisins (erreur pardonnable), en orange sinon
erreurs = pd.crosstab(y_test, y_pred).reindex(index=GENRES, columns=GENRES, fill_value=0)
erreurs = erreurs + erreurs.T

fig = go.Figure()
for a in GENRES:
    for b in GENRES:
        n = erreurs.loc[a, b]
        if GENRES.index(a) < GENRES.index(b) and n >= 8:
            voisin = distance.loc[a, b] == 1
            fig.add_trace(go.Scatter(
                x=[x_genre[a], x_genre[b]], y=[y_genre[a], y_genre[b]], mode="lines",
                line=dict(width=n / 3, color="#2a78d6" if voisin else "#eb6834"),
                showlegend=False,
                hoverinfo="text", text=f"{a} / {b} : {n} erreurs",
            ))
fig.add_trace(go.Scatter(
    x=x_genre, y=y_genre, mode="markers",
    marker=dict(size=[34 if g == CENTRE else 10 for g in GENRES], color="#0b0b0b", opacity=1, line=dict(width=2, color="#fcfcfb")),
    hoverinfo="text", hovertext=GENRES, showlegend=False,
))
fig.add_trace(go.Scatter(x=[0], y=[0], mode="text", text=[CENTRE], textfont=dict(color="#ffffff"),
                         hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(
    x=1.1 * x_genre[CERCLE], y=1.1 * y_genre[CERCLE], mode="text", text=CERCLE,
    textposition=["middle right" if u > 0.2 else "middle left" if u < -0.2 else "middle center" for u in x_genre[CERCLE]],
    hoverinfo="skip", showlegend=False,
))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(width=3, color="#2a78d6"), name="erreur entre voisins"))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(width=3, color="#eb6834"), name="erreur lointaine"))
fig.update_layout(
    title="Confusions du modele retenu sur le test",
    width=750, height=750, plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb", font=dict(color="#0b0b0b"),
    xaxis=dict(visible=False, range=[-1.7, 1.7]), yaxis=dict(visible=False, range=[-1.35, 1.35], scaleanchor="x"),
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.02),
)
fig.show()

Les morceaux ajoutes ont des descripteurs un peu differents de ceux de 2021
(partie 2.5). Si le modele apprend « nouveau pipeline » plutot que le genre,
il se trompe davantage sur l'un des deux groupes.

In [ ]:
par_origine = pd.DataFrame({"origine": etiquetes.loc[y_test.index, "origine"].to_numpy(), "exact": pas == 0, "voisin": pas <= 1})
par_origine.groupby("origine")[["exact", "voisin"]].agg(["size", "mean"]).round(3)

In [ ]:
# Erreurs lointaines (2 pas ou plus), pour juger a l'oreille : changer GENRE et relancer
test = etiquetes.loc[y_test.index, ["Artist", "Song", "origine", "source", "genre"]]
test["predit"] = y_pred
test["pas"] = pas

GENRE = "Rock"
test[(test["genre"] == GENRE) & (test["pas"] >= 2)].head(20)

## 7. Prediction des morceaux sans genre

Le modele retenu donne un genre aux morceaux que Deezer n'a pas classes. Rien
n'est ecrit : pour garder ces predictions, les recopier dans `df["genre"]`
puis relancer la partie 4.

In [ ]:
sans_genre = df[df["genre"].isna()].copy()
sans_genre["genre_predit"] = modele.predict(scaler.transform(sans_genre[descripteurs]))
sans_genre[["Artist", "Song", "playlist", "genre_predit"]]